# TrashScan — Path B

Este notebook assume que a estrutura **já está baixada/criada no volume** do RunPod:

- `/workspace/.venv`
- `/workspace/TrashScan`
- `/workspace/TACO`
- `/workspace/external_datasets`
- `/workspace/processed_4cls`
- `/workspace/runs`

O foco aqui é rodar o fluxo do **Path B** usando os scripts `.py` do projeto, sem refazer downloads e sem refazer merge/preprocessamento por padrão.

## 1) Imports, paths e utilitários

In [ ]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_4cls'

DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'
DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

# Path A: necessário porque o Path B usa o melhor detector treinado no Path A
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'

# Path B: treino do classificador ViT-L
RUNS_PATH_B_DIR = WORKSPACE / 'runs' / 'path_B'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

# Scripts principais
TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'

for p in [RUNS_PATH_B_DIR, MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RUNS_PATH_B_DIR    =', RUNS_PATH_B_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)

    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)

## 2) Verificação dos arquivos principais

In [ ]:
required_paths = [
    TRAIN_PATH_B_SCRIPT,
    PROCESSED_DIR,
    RUNS_PATH_A_DIR,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos/pastas ausentes:")
    for p in missing:
        print(" -", p)
else:
    print("Tudo certo.")

## 3) Configuração de GPU e treino

In [ ]:
if torch.cuda.is_available():
    DEVICE = "0"
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = "cpu"
    gpu_name = "cpu"

print("Device:", DEVICE)
print("GPU:", gpu_name)

EPOCHS = 50
BATCH = 8
LR = 5e-5
PATIENCE = 10
DET_CONF = 0.25

CLASSIFIERS = [
    "vit_l16_imagenet",
]

## 4) Encontrar o melhor peso do Path A

In [ ]:
detector_candidates = [
    RUNS_PATH_A_DIR / "yolov8m" / "weights" / "best.pt",
    RUNS_PATH_A_DIR / "yolov9s" / "weights" / "best.pt",
    RUNS_PATH_A_DIR / "yolov11m" / "weights" / "best.pt",
]

for p in detector_candidates:
    print(p, "->", p.exists())

DETECTOR_WEIGHTS = detector_candidates[0]

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f"Detector não encontrado: {DETECTOR_WEIGHTS}")

print("Usando detector:", DETECTOR_WEIGHTS)

## 5) Conferir estrutura do Path B

In [2]:
for split in ["train", "val", "test"]:
    path_b_dir = PROCESSED_DIR / split / "path_B"
    print(split, path_b_dir, "->", path_b_dir.exists())

    for sub in ["images", "labels", "crops"]:
        p = path_b_dir / sub
        print("  ", sub, "->", p.exists())

NameError: name 'PROCESSED_DIR' is not defined

## 6) Treino Path B — ViT-L fine-tuning

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache"),
])

## 7) Resumo dos treinos

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--output", str(RUNS_PATH_B_DIR),
    "--summarize",
])

## 8) Ver métricas do ViT-L

In [ ]:
metrics_path = RUNS_PATH_B_DIR / "vit_l16_imagenet" / "metrics.json"

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)

    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
else:
    print("metrics.json ainda não encontrado:", metrics_path)